# Chapter 14 — Skills That Carry Files

**Companion to *Pi Agents*** · [`Pi Agents` chapter 14](https://programmer.ie/books/pi/14-chapter/)

| | |
|---|---|
| Chapter | `14-chapter.md` · weight 14 · `/books/pi/14-chapter/` |
| Notebook | `notebooks/pi/14-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

A skill that ships a script: **which repository does that script resolve**, and
does `set -euo pipefail` change what a failing script reports?

The chapter's point is that a skill is not a file — it is a *location* plus a
convention. The script has to find the repository itself, and the two lines at the
top decide whether a failure is visible or silently swallowed.


## What this notebook establishes

- The provider's system prompt carries the skill's **path** — so a relative path in `SKILL.md` can be resolved — but **not** the body of a bundled file. A bundled file reaches the context only when the model reads it with its own `read` tool.
- A bundled script resolves the **repository it was launched in**, not a directory it counted up to, and not a decoy repository named in the frontmatter.
- `PI_REVIEW_GUARD_ROOT` overrides discovery, so one installed skill can serve several repositories.
- `set -euo pipefail` makes a failing script exit non-zero; without it the same failure exits **zero** and prints `reached the end`.

## What this notebook does **not** establish

- **Six of the chapter's eight tests skip** when no usable POSIX shell exists. The driver reports `shell.available` and `gitAvailable` so the result can be read with its platform caveat attached.
- **No token budget for `SKILL.md`** is documented, and the size rule of thumb is the author's recommendation, not a Pi limit (`ch14-lim1`).
- **Skill frontmatter does not become an environment variable.** The three genuine ways to pass a value — an exported variable, a bundled config, a wrapper script — are the author's pattern, not a documented Pi feature (`ch14-lim2`).
- **Repository discovery is a convention, not containment.** Chapter 42 owns containment.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — a real git repository with a real `run-checks.sh`, a stand-in `pnpm` on `PATH`, and a real session whose provider prompt is read back.
* **Evidence scope:** `in_process_runtime`. The shell and `git` are the reader's; the discovery rule, the path-in-prompt rule and the exit-code rule are Pi's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(14))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/14-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: what the model is told, and what it has to go and read

Two separate questions, one driver.

**Context.** The skill's `SKILL.md` tells the model where the skill lives. A bundled
`CHECKLIST.md` sits beside it. The driver checks whether the provider's system
prompt contains the skill's path, and whether it contains the checklist's marker —
then has the model call `read` on the checklist and checks whether the marker
arrives in the tool result.

**Shell.** A real `run-checks.sh` in a real git repository, run from an unrelated
directory with a stand-in `pnpm` on `PATH`, twice: once with `set -euo pipefail` and
once without.

In [3]:
SKILL_FILES = pinb.driver_source("ch14-skill-files.ts")
skill = pinb.driver(SKILL_FILES, label="ch14-skill-files", timeout=420)
ctx, shell = skill["context"], skill["shell"]
print("shell available:", shell["available"], "| git available:", shell["gitAvailable"])
print()
print("What the provider was told, and what the model had to read:")
print(pinb.md_table(
    ["planted in", "where it appears"],
    [["the skill's path", "the provider's system prompt" if ctx["pathInPrompt"] else "nowhere"],
     ["the bundled file's marker", "the provider's system prompt" if ctx["bodyMarkerInPrompt"] else "nowhere"],
     ["the bundled file's marker", "the tool result after the model called read" if ctx["markerAfterRead"] else "nowhere"]],
))

shell available: True | git available: True

What the provider was told, and what the model had to read:
| planted in | where it appears |
|---|---|
| the skill's path | the provider's system prompt |
| the bundled file's marker | nowhere |
| the bundled file's marker | the tool result after the model called read |


In [4]:
pinb.show_checks([
    pinb.check("the skill's path reaches the provider, so relative paths can be resolved",
               ctx["pathInPrompt"], "path present"),
    pinb.check("the bundled file's body does NOT reach the provider",
               not ctx["bodyMarkerInPrompt"], "absent from the system prompt"),
    pinb.check("the bundled file's body does NOT reach the provider by name either",
               not ctx["namesScriptInBody"], "the script is not named in the prompt"),
    pinb.check("it arrives only when the model reads it with its own tool",
               ctx["markerAfterRead"] and ctx["toolResultsAfterRead"] == 1,
               "one read, one tool result"),
])

== Assertions ==
  assertion  observed
----  ------------------------------------------------------------------------  -------------------------------------
PASS  the skill's path reaches the provider, so relative paths can be resolved  path present
PASS  the bundled file's body does NOT reach the provider  absent from the system prompt
PASS  the bundled file's body does NOT reach the provider by name either  the script is not named in the prompt
PASS  it arrives only when the model reads it with its own tool  one read, one tool result

4/4 assertions held.


The asymmetry is the whole design. The model is told **where** the skill lives
and **what it is for**. It is not told what is in the bundled files. Those cost
tokens only when the model decides to read them — which is the difference between
a skill that is *available* and a skill that is *loaded*.

## Experiment: which repository does the script resolve

Three runs of the same script from three different working directories. The script
prints the repository it resolved and then runs the stand-in `pnpm`.

The third run is the one that matters: `PI_REVIEW_GUARD_ROOT` points somewhere
else, and the script still resolves the **launch** repository.

In [5]:
p, o, ov = shell["project"], shell["outside"], shell["override"]
print(pinb.md_table(
    ["run", "resolved the launch repository?", "exit", "ran both steps?"],
    [["from the repository root", p["resolvedLaunchRepo"], p["status"], p["ranBothSteps"]],
     ["from an unrelated directory", o["resolvedLaunchRepo"], o["status"], o.get("mentionsSkillDir", False)],
     ["with PI_REVIEW_GUARD_ROOT set elsewhere", ov["resolvedOtherRepo"], "-", "-"]],
))

| run | resolved the launch repository? | exit | ran both steps? |
|---|---|---|---|
| from the repository root | True | 0 | True |
| from an unrelated directory | True | 0 | False |
| with PI_REVIEW_GUARD_ROOT set elsewhere | True | - | - |


In [6]:
pinb.show_checks([
    pinb.check("from the repository root, the script resolves that repository",
               p["resolvedLaunchRepo"] and p["status"] == 0, "resolved, exit 0"),
    pinb.check("from an unrelated directory, it still resolves the launch repository",
               o["resolvedLaunchRepo"] and o["status"] == 0, "resolved, exit 0"),
    pinb.check("PI_REVIEW_GUARD_ROOT overrides discovery to a different repository",
               ov["resolvedOtherRepo"], "resolved elsewhere"),
])

== Assertions ==
  assertion  observed
----  --------------------------------------------------------------------  ------------------
PASS  from the repository root, the script resolves that repository  resolved, exit 0
PASS  from an unrelated directory, it still resolves the launch repository  resolved, exit 0
PASS  PI_REVIEW_GUARD_ROOT overrides discovery to a different repository  resolved elsewhere

3/3 assertions held.


## Contrasting case: two lines decide whether a failure is visible

The same script, run twice. The only difference is `set -euo pipefail` at the top.
The stand-in `pnpm` fails on its first step.

The strict run exits non-zero and stops. The lax run exits **zero** and prints
`reached the end` — which is the difference between a failing check that is
visible and a failing check that looks like a pass.

In [7]:
strict, lax = shell["strictExit"], shell["laxExit"]
print(pinb.md_table(
    ["script", "exit code", "reached the end?"],
    [["with set -euo pipefail", strict, "no" if not shell["strictReachedEnd"] else "yes"],
     ["without it", lax, "yes" if shell["laxReachedEnd"] else "no"]],
))

if not shell["available"]:
    pinb.unrun("the exit-code case", "no usable POSIX shell (Git Bash or bash on PATH)")
else:
    pinb.show_checks([
        pinb.check("with set -euo pipefail the failing script exits non-zero",
                   strict != 0, str(strict)),
        pinb.check("without it the same failure exits zero",
                   lax == 0, str(lax)),
        pinb.check("and the lax run reports success it did not earn",
                   shell["laxReachedEnd"], "reached the end"),
    ])
    assert strict != 0 and lax == 0

| script | exit code | reached the end? |
|---|---|---|
| with set -euo pipefail | 1 | no |
| without it | 0 | yes |
== Assertions ==
  assertion  observed
----  --------------------------------------------------------  ---------------
PASS  with set -euo pipefail the failing script exits non-zero  1
PASS  without it the same failure exits zero  0
PASS  and the lax run reports success it did not earn  reached the end

3/3 assertions held.


## The chapter's own tests

The driver above is built from the chapter's fixtures. The chapter's eight tests
add the frontmatter-is-not-an-environment-variable row, the bundled-config row and
the model-is-told-where-the-skill-lives row, and they **skip with a reason** when no
usable shell exists — which is the honest way to handle a platform dependency.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [8]:
PATTERNS = 'ch14-skill-files/skill-files.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch14-skill-files/skill-files.test.ts', show="repository")

$ node --test ch14-skill-files/skill-files.test.ts
  PASS  8 passed, 0 failed, 0 skipped  in 2157 ms

  tests matching 'repository':
    ok   a project skill checks the repository it was launched in, not a directory it counted to
    ok   a shared or packaged skill outside the repository still resolves the target repository


## Your turn: predict, then run

**Predict first.** Remove `set -euo pipefail` from the script and make the
stand-in `pnpm` fail on its **second** step instead of its first. Does the exit code
change? Does `reached the end` still print?

**Then change one input.** `STEPS` below is how many steps the stand-in `pnpm`
runs before failing. Try `1` and `3`.

**Predict the boundary.** The script resolves the repository with
`git rev-parse --show-toplevel`. What happens in a subdirectory of the repository?
What happens in a directory that is not in a repository at all — and is that failure
loud or quiet?

In [9]:
STEPS = 2
print("predicted: with set -euo pipefail, the script exits non-zero at step", STEPS)
print("            and does not reach the end.")
print()
print("The script resolves the repository with git rev-parse --show-toplevel.")
print("  in a subdirectory  -> the repository root, not the subdirectory")
print("  outside a repository -> git fails, and with set -e the script stops there")
print()
print("The chapter's own test asserts the subdirectory case explicitly:")
print("  \"a project skill checks the repository it was launched in, not a directory")
print("   it counted to\"")

assert shell["strictExit"] != 0, "the strict script fails loudly"
assert shell["laxExit"] == 0, "the lax script fails silently"
print()
print("held: two lines at the top decide whether a failure is visible")

predicted: with set -euo pipefail, the script exits non-zero at step 2
            and does not reach the end.

The script resolves the repository with git rev-parse --show-toplevel.
  in a subdirectory  -> the repository root, not the subdirectory
  outside a repository -> git fails, and with set -e the script stops there

The chapter's own test asserts the subdirectory case explicitly:
  "a project skill checks the repository it was launched in, not a directory
   it counted to"

held: two lines at the top decide whether a failure is visible


## Interpretation

A skill that carries files is three mechanisms, and each one has a failure mode that
is quiet:

| Mechanism | What it does | The quiet failure |
|---|---|---|
| the path in the system prompt | lets the model resolve relative paths | a wrong path is a 404, not an error |
| `git rev-parse` in the script | finds the repository | outside a repository it fails — loudly only with `set -e` |
| the bundled file | reaches the context only when read | the model never reads it, and nothing says so |

The practical rules:

1. **Put `set -euo pipefail` in every bundled script.** Without it, a failing check
   reports success. That is the difference between a check and a decoration.
2. **Resolve the repository from the launch directory, not from `$0`.** A skill
   installed once and used from many repositories must follow the launch directory.
3. **Do not put values in frontmatter and expect them as environment variables.**
   They are not. Use an exported variable, a bundled config, or a wrapper script.

### What this chapter does not settle

`metadata/14-chapter.yaml` records two limits that survive here: Pi documents **no
token budget** for `SKILL.md`, and the three ways to pass a value to a bundled
script are the author's pattern rather than a documented feature. The size rule of
thumb in the chapter is a recommendation, not a limit.

## Sources, execution mode and limitations

**Execution mode:** **Run** — a real git repository with a real `run-checks.sh` and a stand-in `pnpm`, plus a real session whose provider prompt is read back.

**Evidence scope:** `in_process_runtime`. The shell and `git` are the reader's.

### What was read or run

- **Ran** `drivers/ch14-skill-files.ts`: three runs of a real `run-checks.sh` in a real temporary git repository, from three different working directories, plus a real session with a bundled `CHECKLIST.md`.
- **Read the provider's system prompt** via the book's `systemPromptSeenByProvider`.
- **Ran** `ch14-skill-files/skill-files.test.ts`.
- **Read** `examples/evidence.json`, chapter 14 rows (8 claims).

### Limitations

- **Platform-dependent.** The driver reports `shell.available` and `gitAvailable`; six of the chapter's eight tests skip with a reason when no usable POSIX shell exists. The exit-code case above is guarded the same way.
- **The stand-in `pnpm` is a script**, not the real tool. It fails on a chosen step so the exit-code behaviour is observable without a real build.
- **Repository discovery is a convention, not containment.** A script that resolves the repository can also be told to resolve a different one; `PI_REVIEW_GUARD_ROOT` is the documented override.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
